In [1]:
%load_ext autoreload
%autoreload 2

from math import pi
import time

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import xarray as xr

from fluxoniumcr import DATA_DIR
from fluxoniumcr.dressed_control_fluxonium import create_driven_fluxonium
from fluxoniumcr.qubits.fluxonium import Fluxonium
from fluxoniumcr.utils import load_arguments

In [2]:
# parent_path = DATA_DIR/"charge_operator/EJ=4.00,EC=1.20,EL=0.40"
parent_path = DATA_DIR/"charge_operator/EJ=5.60,EC=1.87,EL=0.56"

argm = load_arguments(parent_path)

dataset_path = parent_path/"driven_charge_operator.hdf5"

In [3]:
fx = Fluxonium(
    EJ=argm.EJ,
    EC=argm.EC,
    EL=argm.EL,
    dim=argm.dim,
    cutoff=argm.cutoff
)
qubit_frequency = fx.eigenvalues[1] - fx.eigenvalues[0]
n_op = fx.get_operator('charge')
Ω0 = qubit_frequency/abs(n_op[0, 1])

lookup_amps = Ω0 * argm.lookup_amplitudes
lookup_deg_tol = argm.lookup_degeneracy_tol

In [44]:
if dataset_path.exists():
    dataset = xr.load_dataset(dataset_path)
else:
    dataset = xr.Dataset(
        attrs=dict(
            EJ=argm.EJ,
            EC=argm.EC,
            EL=argm.EL,
            lookup_degeneracy_tolerance=lookup_deg_tol,
            phase_gauge=argm.phase_gauge,
        )
    )

    frequency_coord = xr.DataArray(
        argm.frequencies,
        dims=['frequency'],
        attrs=dict(
            long_name="Drive frequency",
            units="Grad/s",
        )
    )

    amplitude_coord = xr.DataArray(
        Ω0 * argm.amplitudes,
        dims=['amplitude'],
        attrs=dict(
            long_name="Drive amplitude",
            units="Grad/s",
            plot_unit=Ω0,
        )
    )
    
    harmonic_data = np.fft.fftfreq(
        argm.n_harmonics, 1/argm.n_harmonics
    ).astype(np.int32)
    harmonic_coord = xr.DataArray(
        # Keep only odd harmonics
        harmonic_data[(harmonic_data%2).astype(bool)],
        dims=['harmonic'],
        attrs=dict(
            long_name="Harmonic index"
        )
    )
    
    level_data = np.arange(argm.truncated_dim, dtype=np.uint8)
    bra_coord = xr.DataArray(level_data, dims=['bra'])
    ket_coord = xr.DataArray(level_data, dims=['ket'])
    
    dataset['quasienergy'] = xr.DataArray(
        np.float32('nan'),
        coords=[
            frequency_coord,
            amplitude_coord,
            ket_coord,
        ],
        attrs=dict(
            long_name="Floquet quasienergy"
        )
    )
    dataset['matrix'] = xr.DataArray(
        np.float32('nan'),
        coords=[
            frequency_coord,
            amplitude_coord,
            harmonic_coord,
            bra_coord,
            ket_coord,
        ],
        attrs=dict(
            long_name="Charge matrix element"
        )
    )

In [45]:
last_save = time.time()


for frequency in tqdm(dataset.frequency.data):
    floquet_basis = None
    
    for amplitude in dataset.amplitude.data:
        ds = dataset.loc[dict(
            frequency=frequency,
            amplitude=amplitude,
        )]
        
        # Check if matrix elements have already been calculated.
        if not np.isnan(ds.matrix.data.ravel()[0]): continue
        
        if floquet_basis is None:
            floquet_basis = create_driven_fluxonium(
                fx,
                drive_freq=frequency,
                N=argm.n_steps,
                phase_gauge=argm.phase_gauge,
            )
            floquet_basis.generate_lookup(
                lookup_amps,
                deg_tol=lookup_deg_tol,
            )
        
        result = floquet_basis.query(amplitude)
        evals = result.freqs
        dressed_n_op = result.dress_fft(n_op)
        
        ds.quasienergy[:] = evals[:len(ds.ket)]
        
        # Real elements are all zero.
        ds.matrix[:] = dressed_n_op[ds.harmonic, :len(ds.ket), :len(ds.bra)].imag
    
    # Save every 5 mins
    if (now := time.time()) - last_save > 5*60:
        dataset.to_netcdf(dataset_path)
        last_save = now
        
dataset.to_netcdf(dataset_path)

100%|█████████████████████████████████████████████████████| 1/1 [00:19<00:00, 19.97s/it]
